# PRISM step 2: what are the columns?

Reads a file (step 1), proposes which columns are the row IDs, the measurements and the annotations, with a reason for each decision, and gives the measurements as a numeric matrix (features x samples).

In [ ]:
from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q --force-reinstall --no-deps "git+https://{token}@github.com/Isra-inventor/PRISM.git@claude/prism-data-input-recognition-9adu1r"
import prism
print("PRISM", prism.__version__)
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/PRISM_data"

## 1. Read the file and look at the proposed layout

In [ ]:
table = prism.read(f"{DATA}/SomaExpr_common.csv")   # change the file name to yours
layout = prism.propose_layout(table)
layout

## 2. Check the role of every column
Look in particular at numeric columns: are the `measurement` ones really samples?

In [ ]:
roles = layout.roles()
roles[roles.kind == "numeric"]

## 3. Correct anything that is wrong
For example, a reference column that follows the samples but is not a sample:
```python
layout = layout.set(annotations=["Median Normal Plasma (RFU)"])
```
Other options: `set(orientation="samples_in_rows")`, `set(id_column="SeqId")`, `set(measurements=[...])`.

In [ ]:
# layout = layout.set(annotations=[...])
layout

## 4. The measurements as numbers (features x samples)
Missing cells stay empty (NaN); nothing is filled.

In [ ]:
X = layout.matrix()
print(X.shape, "missing cells:", int(X.isna().sum().sum()))
X.iloc[:5, :6]

In [ ]:
layout.annotations().head()

## 5. The second file
The metabolomics file has one row per sample; the layout turns it around so the matrix is still features x samples.

In [ ]:
metab = prism.propose_layout(prism.read(f"{DATA}/MetaboExpr_common.csv"))
metab

In [ ]:
metab.matrix().iloc[:5, :6]